In [ ]:
!pip install pyhealth

In [29]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pyhealth.datasets import MIMIC3Dataset

dataset = MIMIC3Dataset(
    root='/content/drive/MyDrive/mimic-iii-clinical-database-1.4 3',
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    dev= False
)

dataset.stats()

In [32]:
# Helper functions
from pyhealth.data import Patient, Event
from datetime import datetime


def getDOB(P: Patient):
  return P.get_events(event_type='patients')[0].attr_dict['dob']


def getDischTime(E: Event):
  return E.attr_dict['dischtime']

def convertToDT(dtString: str):
  format = '%Y-%m-%d %H:%M:%S'
  dt = datetime.strptime(dtString, format)
  return dt

def getAge(dob: datetime, admissionDate: datetime):
  age = (admissionDate - dob).days // 365
  return age

def getHADID(E: Event):
  return E.attr_dict['hadm_id']


def get_time_gap_bucket(days):
    if days < 7:
        return "gap_0_7"
    elif days < 30:
        return "gap_7_30"
    elif days < 90:
        return "gap_30_90"
    elif days < 365:
        return "gap_90_365"
    else:
        return "gap_365_plus"



In [40]:
from collections import defaultdict
from pyhealth.tasks import BaseTask
from pyhealth.data import Patient
from typing import List, Dict
from pyhealth.processors import BinaryLabelProcessor, SequenceProcessor, NestedSequenceProcessor
import polars as pl


class BEHRTTask(BaseTask):
    task_name = "BEHRT collection task v2"
    input_schema: Dict[str, str] = {
        "codes": NestedSequenceProcessor,
        "time_gaps": SequenceProcessor,
        "ages": SequenceProcessor,
    }
    output_schema: Dict[str, str] = {
        "readmission": BinaryLabelProcessor,
    }

    # CHANGE THIS
    # OPTIMIZATION HYPERPARAMS
    # This can be placed insdie the constructor itself (you guys can do that if you want)
    MAX_VISITS = 30
    MAX_CODES_PER_VISIT = 30

    def pre_filter(self, df: pl.LazyFrame) -> pl.LazyFrame:

        has_enough_visits = (
            df.filter(pl.col("event_type") == "admissions")
            .group_by("patient_id")
            .agg(pl.count().alias("visit_count"))
            .filter(pl.col("visit_count") >= 3)
            .select("patient_id")
            .collect()
            .to_series()
        )

        has_diagnoses = (
            df.filter(pl.col("event_type") == "diagnoses_icd")
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        has_procedures = (
            df.filter(pl.col("event_type") == "procedures_icd")
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        has_dob = (
            df.filter(
                (pl.col("event_type") == "patients")
                & (pl.col("patients/dob").is_not_null())
            )
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        died_in_hospital = (
            df.filter(
                (pl.col("event_type") == "admissions")
                & (pl.col("admissions/hospital_expire_flag") == "1")
            )
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        filtered_df = df.filter(
            pl.col("patient_id").is_in(has_enough_visits)
            & (pl.col("patient_id").is_in(has_diagnoses) | pl.col("patient_id").is_in(has_procedures))
            & pl.col("patient_id").is_in(has_dob)
            & ~pl.col("patient_id").is_in(died_in_hospital)
        )

        return filtered_df

    def __call__(self, patient: Patient) -> List[Dict]:
        samples = []

        dob_dt = convertToDT(getDOB(patient))

        diagnoses = patient.get_events(event_type='diagnoses_icd')
        procedures = patient.get_events(event_type='procedures_icd')
        admissions = patient.get_events(event_type='admissions')

        visit_codes = defaultdict(list)
        for diag in diagnoses:
            visit_codes[getHADID(diag)].append(diag.attr_dict['icd9_code'])
        for proc in procedures:
            visit_codes[getHADID(proc)].append(proc.attr_dict['icd9_code'])

        for vid in visit_codes:
            visit_codes[vid] = visit_codes[vid][:self.MAX_CODES_PER_VISIT]


        sorted_admissions = sorted(admissions, key=lambda e: e.timestamp)

        sorted_admissions = sorted_admissions[-self.MAX_VISITS:]

        for i, admission in enumerate(sorted_admissions):

            codes_so_far = []
            ages_so_far = []
            time_gaps_so_far = []

            for j in range(i + 1):
                vid = getHADID(sorted_admissions[j])
                codes_so_far.append(visit_codes.get(vid, []))

                age = getAge(dob_dt, sorted_admissions[j].timestamp)
                ages_so_far.append(str(age))

                if j == 0:
                    time_gaps_so_far.append("gap_none")
                else:
                    prev_discharge = convertToDT(getDischTime(sorted_admissions[j - 1]))
                    curr_admit = sorted_admissions[j].timestamp
                    gap_days = (curr_admit - prev_discharge).days
                    time_gaps_so_far.append(get_time_gap_bucket(gap_days))

            if i < len(sorted_admissions) - 1:
                discharge = convertToDT(getDischTime(admission))
                next_admit = sorted_admissions[i + 1].timestamp
                gap = (next_admit - discharge).days
                readmission = 1 if gap <= 30 else 0
            else:
                readmission = 0

            samples.append({
                "patient_id": patient.patient_id,
                "codes": codes_so_far,
                "ages": ages_so_far,
                "time_gaps": time_gaps_so_far,
                "readmission": readmission,
            })

        return samples

In [41]:
from pyhealth.models import BaseModel
from pyhealth.models.embedding import EmbeddingModel
import torch
import torch.nn as nn
import numpy as np
import math

In [42]:
# I pulled this from the BEHRT paper's Github
# To sum it shortly, they use a sinusodial function to denote the relevancy scores
# of each position

class CustomPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i]     = math.sin(pos / (10000 ** ((2 * i) / d_model)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * (i + 1)) / d_model)))
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [43]:

class BEHRT(BaseModel):
  def __init__(self, dataset, embedding_dim=128, num_heads=4,
                 num_layers=2, dropout=0.5, max_len=512, **kwargs):
        super().__init__(dataset=dataset, **kwargs)

        self.label_key = self.label_keys[0]


        code_embed_size = dataset.input_processors["codes"].vocab_size()    # 707
        age_embed_size = dataset.input_processors["ages"].vocab_size()      # 43
        gap_embed_size = dataset.input_processors["time_gaps"].vocab_size()  # 8



        self.cls_token = nn.Parameter(torch.zeros(1, 1, embedding_dim))
        nn.init.normal_(self.cls_token, std=0.02)

        self.code_embedding = nn.Embedding(code_embed_size, embedding_dim, padding_idx=0)
        self.age_embedding = nn.Embedding(age_embed_size, embedding_dim, padding_idx=0)
        self.gap_embedding = nn.Embedding(gap_embed_size, embedding_dim, padding_idx=0)
        self.pos_embedding = CustomPositionalEmbedding(d_model=embedding_dim, max_len=max_len+1)




        # Transformer Encoder

        encoder = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(encoder_layer= encoder, num_layers=num_layers)
        self.fc = nn.Linear(embedding_dim, self.get_output_size())


  def forward(self, **kwargs):
    y_true = kwargs[self.label_key].to(self.device)


    codes = kwargs['codes']
    ages = kwargs['ages']
    time_gaps = kwargs['time_gaps']

    batch, max_visits, max_codes = codes.shape
    seq_len = max_codes * max_visits


    # Match the dimensionality of the Tensors

    codes_flat = codes.reshape(batch, seq_len)
    ages_flat = ages.unsqueeze(-1).expand(-1, -1, max_codes).reshape(batch, seq_len)
    gaps_flat = time_gaps.unsqueeze(-1).expand(-1, -1, max_codes).reshape(batch, seq_len)




    codes_embedding = self.code_embedding(codes_flat)
    ages_embedding = self.age_embedding(ages_flat)
    gaps_embedding = self.gap_embedding(gaps_flat)


    x = codes_embedding + ages_embedding + gaps_embedding


    cls_tokens = self.cls_token.expand(batch, 1, -1)       # [B, 1, D]
    x = torch.cat([cls_tokens, x], dim=1)                  # [B, seq_len + 1, D]

    pos_embed = self.pos_embedding(x)

    # Padding mask for model
    token_padding_mask = codes_flat == 0


    cls_padding_mask = torch.zeros(
        batch,
        1,
        dtype=torch.bool,
        device=self.device,
    )                                                      # [B, 1]

    padding_mask = torch.cat(
        [cls_padding_mask, token_padding_mask],
        dim=1,
    )


    h = self.transformer(x, src_key_padding_mask=padding_mask)
    # h: (batch, seq_len + 1, embedding_dim)

    cls_h = h[:, 0, :]                  # (batch, embedding_dim) — pull CLS
    logit = self.fc(cls_h)




    return {
        "loss": self.get_loss_function()(logit, y_true),
        "y_prob": self.prepare_y_prob(logit) ,
        "y_true": y_true,
        "logit": logit,
    }



In [44]:
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.trainer import Trainer

In [ ]:
behrt_readmission_task = BEHRTTask()

# Setting the new custom task to the dataset
sample_dataset = dataset.set_task(behrt_readmission_task)

# Print out for clarity
print(f"Total samples: {len(sample_dataset)}")
print(sample_dataset[0])

In [47]:
# STEP 3: Split and create dataloaders
train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

Testing of the actual model you can skip this section if you want to these are just debug logs

In [ ]:
#Instantiation Test

from pyhealth.datasets import MIMIC3Dataset, SampleDataset



model = BEHRT(dataset=sample_dataset, max_len=1096)

print(model)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import DataLoader
from pyhealth.datasets import get_dataloader

# Create a small dataloader
loader = get_dataloader(sample_dataset, batch_size=4, shuffle=False)

# Grab one batch
batch = next(iter(loader))

# Inspect what's in it
print("Batch keys:", batch.keys())
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: type={type(v)}")

In [ ]:
# Forward Pass Test
model.eval()
with torch.no_grad():
    output = model(**batch)

print("Output keys:", output.keys())
for k, v in output.items():
    if hasattr(v, "shape"):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
        if k in ("loss", "y_prob", "logit"):
            print(f"    sample values: {v.flatten()[:4].tolist()}")
    else:
        print(f"  {k}: {v}")

In [51]:
# Backward pass

model.train()
batch = next(iter(loader))   # fresh batch, just to be safe

output = model(**batch)
loss = output["loss"]
print(f"Loss before backward: {loss.item():.4f}")

loss.backward()

# Check gradient flow
no_grad = []
zero_grad = []
healthy = []

for name, param in model.named_parameters():
    if param.grad is None:
        no_grad.append(name)
    elif param.grad.abs().max().item() == 0:
        zero_grad.append(name)
    else:
        healthy.append(name)

print(f"Healthy gradients:  {len(healthy)} parameters")
print(f"Zero gradients:     {len(zero_grad)} parameters")
print(f"Missing gradients:  {len(no_grad)} parameters")

if no_grad:
    print(f"\n⚠️  Parameters with no grad:")
    for n in no_grad:
        print(f"   {n}")
if zero_grad:
    print(f"\n⚠️  Parameters with zero grad:")
    for n in zero_grad:
        print(f"   {n}")

Loss before backward: 0.6401
Healthy gradients:  30 parameters
Zero gradients:     0 parameters
Missing gradients:  1 parameters

⚠️  Parameters with no grad:
   _dummy_param


In [52]:
# I strongly advise running this

# This will give you an idea of what to set as the hidden dimension
#If you scroll above i set mine to 1096

# Run this first and check what you get for the worst case if you plan to change
# the Hyperparam configs I did in the custom task


max_visits_overall = 0
max_codes_overall = 0
worst_seq_len = 0

for sample in sample_dataset:
    codes = sample["codes"]                          # nested list: visits × codes
    n_visits = len(codes)
    max_codes_in_sample = max((len(v) for v in codes), default=0)
    seq_len = n_visits * max_codes_in_sample

    if n_visits > max_visits_overall:
        max_visits_overall = n_visits
    if max_codes_in_sample > max_codes_overall:
        max_codes_overall = max_codes_in_sample
    if seq_len > worst_seq_len:
        worst_seq_len = seq_len

print(f"Max visits in any sample:     {max_visits_overall}")
print(f"Max codes in any single visit: {max_codes_overall}")
print(f"Worst per-sample seq_len:     {worst_seq_len}")
print(f"Theoretical worst-case batch: {max_visits_overall * max_codes_overall}")
print(f"\nSet max_len to: {max_visits_overall * max_codes_overall + 1}  (+1 for CLS)")

Max visits in any sample:     30
Max codes in any single visit: 30
Worst per-sample seq_len:     900
Theoretical worst-case batch: 900

Set max_len to: 901  (+1 for CLS)


Now Time to evaluate the model and see how it performs.


In [ ]:
# This will take some time Be patient
# Id pay attention to the CPU and GPU usage first but it shouldnt be an issue
# with the optimization params set in place it should be fine
# took me 12 mins to run (maybe faster if ur on PC or ur laptop is plugged in)
# Just depends on ur CPU and GPU usage



# I didnt set the seed to 42 btw so make sure u guys do that before anything

trainer = Trainer(model=model)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=1,
    monitor="roc_auc"
)

trainer.evaluate(test_dataloader)